# DataLab: Movie Box Office Worldwide Revenue Prediction
### Comprehensive End-to-End Machine Learning Pipeline & Exploratory Analysis

---

## Section 1: Project Introduction & Supervised Learning Formulation

### 1.1 Problem Statement & Objectives
Predicting worldwide theatrical box-office revenue is one of the most commercially critical and quantitatively challenging problems in the entertainment analytics industry. Film production, licensing, and distribution demand substantial upfront capital expenditure—often ranging from tens of millions to hundreds of millions of dollars—committed years before theatrical debut. Being able to estimate potential box-office returns from pre-release metadata provides studio executives, investors, and streaming platforms with empirical risk assessment and portfolio optimization tools.

### 1.2 Mathematical Formulation
We formulate box-office prediction as a **supervised regression task**:
$$\hat{y} = f(X; \theta)$$
where:
* **$X \in \mathbb{R}^{n \times d}$** is the pre-release feature matrix comprising 37 engineered numerical, temporal, categorical, and binary metadata attributes.
* **$y \in \mathbb{R}^n$** is the ground-truth worldwide box-office revenue in USD (`revenue`).
* **$f(\cdot)$** is a predictive regression estimator parameterized by $\theta$.

### 1.3 Target Transformation: Why $\log(1 + y)$?
Worldwide box-office revenue spans nearly eight orders of magnitude (from under \$10,000 to over \$1.5 billion) and exhibits extreme right skewness (long Pareto-like tail). Directly optimizing mean squared error (MSE) on raw dollars causes gradients to be completely dominated by rare billion-dollar blockbusters while ignoring mid-budget and independent releases. 
Consequently, we transform our target into log-space:
$$z = \log(1 + y) = \text{log1p}(y)$$
All regression algorithms are trained to predict $z$. Inferences $\hat{z}$ are subsequently mapped back to dollar amounts via the natural exponential inverse:
$$\hat{y} = \max\left(0, \exp(\hat{z}) - 1\right) = \max\left(0, \text{expm1}(\hat{z})\right)$$

### 1.4 Evaluation Metrics
Because Kaggle's TMDB Box Office Prediction competition evaluates submissions using Root Mean Squared Logarithmic Error, our primary metric is:
$$\text{RMSLE}(y, \hat{y}) = \sqrt{\frac{1}{n} \sum_{i=1}^n \left(\log(1 + \hat{y}_i) - \log(1 + y_i)\right)^2}$$
We also monitor:
* **Mean Absolute Error (MAE):** Average absolute dollar deviation: $\frac{1}{n}\sum |y_i - \hat{y}_i|$.
* **Coefficient of Determination ($R^2$):** Proportion of revenue variance explained by the model: $1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$.

### 1.5 Strict Pre-Release Integrity
A valid machine learning pipeline must respect real-world causality: **only information available before a film's public release may be used**. Post-release signals such as user review ratings (`vote_average`), vote counts (`vote_count`), and dynamic popularity indices (`popularity`) reflect audience reception after theatrical debut. Including them constitutes severe data leakage and invalidates predictive realism.

---
## Section 2: Environment Setup & Package Requirements

This notebook is **100% self-contained and standalone**. It does not import or depend on any local project modules or files from `src/` or `app.py`. All parsing, feature extraction, preprocessing, model architectures, and evaluation functions are implemented directly within this notebook.

### Required Dependencies
The following standard Python data science libraries are required:
```bash
pip install pandas numpy scikit-learn plotly matplotlib joblib
```

In [1]:
# Standard Library Imports
import os
import sys
import ast
import json
import re
import time
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

# Numerical & Scientific Computing
import numpy as np
import pandas as pd

# Visualization Libraries
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Scikit-Learn Preprocessing & Modeling
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, r2_score

# Model Persistence
import joblib

# Global Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

# Set interactive plotting defaults
pio.templates.default = "plotly_white"

print(f"Python Environment: {sys.version.split()[0]}")
print("All core dependencies successfully loaded. Zero project-specific module imports.")

Python Environment: 3.12.0
All core dependencies successfully loaded. Zero project-specific module imports.


---
## Section 3: Dataset Loading & Directory Resolution

We implement `load_dataset()` from scratch. To ensure this notebook executes seamlessly across different environments (local clone, nested `notebooks/` directory, Kaggle kernel, or Google Colab), the resolution engine searches the current working directory, parent directory, and standard subdirectories (`data/`, `tmdb-box-office-prediction/`).

In [2]:
def load_dataset(filename: str = "train.csv", filepath: Optional[str] = None) -> Tuple[Optional[pd.DataFrame], str]:
    """
    Searches for and loads the TMDB dataset CSV file from multiple standard candidate locations.
    
    Parameters:
        filename: Name of the target file ('train.csv', 'test.csv', etc.)
        filepath: Explicit override path if provided.
        
    Returns:
        Tuple of (DataFrame, status_message)
    """
    if filepath and os.path.isfile(filepath):
        df = pd.read_csv(filepath)
        return df, f"Successfully loaded {len(df):,} rows from explicit path: {filepath}"
    
    # Candidate search roots
    cwd = Path.cwd().resolve()
    candidate_dirs = [
        cwd,
        cwd.parent,
        cwd / "data",
        cwd.parent / "data",
        cwd / "tmdb-box-office-prediction",
        cwd.parent / "tmdb-box-office-prediction",
    ]
    
    for candidate_dir in candidate_dirs:
        target_path = candidate_dir / filename
        if target_path.is_file():
            df = pd.read_csv(target_path)
            return df, f"Successfully loaded {len(df):,} rows from {target_path.resolve()}"
            
    return None, f"Error: '{filename}' not found in candidate paths: {[str(d) for d in candidate_dirs]}"

# Load both training and testing datasets
train_df, train_status = load_dataset("train.csv")
test_df, test_status = load_dataset("test.csv")

print(train_status)
print(test_status)

# Verify schema requirements
assert train_df is not None, "Critical: train.csv could not be loaded."
assert test_df is not None, "Critical: test.csv could not be loaded."
assert "revenue" in train_df.columns, "Critical: 'revenue' target column missing from train.csv."
assert "revenue" not in test_df.columns, "Notice: 'revenue' is correctly absent from test.csv (competition test split)."

print(f"\nTraining Set Dimensions: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns")
print(f"Testing Set Dimensions:  {test_df.shape[0]:,} rows × {test_df.shape[1]} columns")

print("\nFirst 5 Rows of Training Set:")
display(train_df.head(5))

Successfully loaded 3,000 rows from D:\IDSE_Project\data\train.csv
Successfully loaded 4,398 rows from D:\IDSE_Project\data\test.csv

Training Set Dimensions: 3,000 rows × 23 columns
Testing Set Dimensions:  4,398 rows × 22 columns

First 5 Rows of Training Set:


,id,belongs_to_collection,budget,genres,homepage,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,runtime,spoken_languages,status,tagline,title,Keywords,cast,crew,revenue
0,1,"[{'id': 313576, 'name': 'Hot Tub Time Machine ...",14000000,"[{'id': 35, 'name': 'Comedy'}]",NaN,tt2637294,en,Hot Tub Time Machine 2,"When Lou, who has become the ""father of the In...",6.58,/tQtWuwvMf0hCc2QR2tkolwl7c3c.jpg,"[{'name': 'Paramount Pictures', 'id': 4}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",2/20/15,93.00,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The Laws of Space and Time are About to be Vio...,Hot Tub Time Machine 2,"[{'id': 4379, 'name': 'time travel'}, {'id': 9...","[{'cast_id': 4, 'character': 'Lou', 'credit_id...","[{'credit_id': '59ac067c92514107af02c8c8', 'de...",12314651
1,2,"[{'id': 107674, 'name': 'The Princess Diaries ...",40000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,tt0368933,en,The Princess Diaries 2: Royal Engagement,Mia Thermopolis is now a college graduate and ...,8.25,/w9Z7A0GHEhIp7etpj0vyKOeU1Wx.jpg,"[{'name': 'Walt Disney Pictures', 'id': 2}]","[{'iso_3166_1': 'US', 'name': 'United States o...",8/6/04,113.00,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,It can take a lifetime to find true love; she'...,The Princess Diaries 2: Royal Engagement,"[{'id': 2505, 'name': 'coronation'}, {'id': 42...","[{'cast_id': 1, 'character': 'Mia Thermopolis'...","[{'credit_id': '52fe43fe9251416c7502563d', 'de...",95149435
2,3,NaN,3300000,"[{'id': 18, 'name': 'Drama'}]",http://sonyclassics.com/whiplash/,tt2582802,en,Whiplash,"Under the direction of a ruthless instructor, ...",64.30,/lIv1QinFqz4dlp5U4lQ6HaiskOZ.jpg,"[{'name': 'Bold Films', 'id': 2266}, {'name': ...","[{'iso_3166_1': 'US', 'name': 'United States o...",10/10/14,105.00,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,The road to greatness can take you to the edge.,Whiplash,"[{'id': 1416, 'name': 'jazz'}, {'id': 1523, 'n...","[{'cast_id': 5, 'character': 'Andrew Neimann',...","[{'credit_id': '54d5356ec3a3683ba0000039', 'de...",13092000
3,4,NaN,1200000,"[{'id': 53, 'name': 'Thriller'}, {'id': 18, 'n...",http://kahaanithefilm.com/,tt1821480,hi,Kahaani,Vidya Bagchi (Vidya Balan) arrives in Kolkata ...,3.17,/aTXRaPrWSinhcmCrcfJK17urp3F.jpg,NaN,"[{'iso_3166_1': 'IN', 'name': 'India'}]",3/9/12,122.00,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,NaN,Kahaani,"[{'id': 10092, 'name': 'mystery'}, {'id': 1054...","[{'cast_id': 1, 'character': 'Vidya Bagchi', '...","[{'credit_id': '52fe48779251416c9108d6eb', 'de...",16000000
4,5,NaN,0,"[{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...",NaN,tt1380152,ko,마린보이,Marine Boy is the story of a former national s...,1.15,/m22s7zvkVFDU9ir56PiiqIEWFdT.jpg,NaN,"[{'iso_3166_1': 'KR', 'name': 'South Korea'}]",2/5/09,118.00,"[{'iso_639_1': 'ko', 'name': '한국어/조선말'}]",Released,NaN,Marine Boy,NaN,"[{'cast_id': 3, 'character': 'Chun-soo', 'cred...","[{'credit_id': '52fe464b9251416c75073b43', 'de...",3923970


---
## Section 4: Dataset Profiling & Understanding

Prior to feature extraction, we inspect column types, missingness patterns, duplicate records, and descriptive statistics across the raw training dataset.

In [3]:
# 4.1 Missing Values and Data Types Audit
missing_series = train_df.isna().sum()
missing_pct = (missing_series / len(train_df)) * 100

data_profile = pd.DataFrame({
    "Data Type": train_df.dtypes.astype(str),
    "Non-Null Count": train_df.notna().sum(),
    "Missing Count": missing_series,
    "Missing Pct (%)": missing_pct.round(2),
    "Unique Values": train_df.nunique()
}).sort_values(by="Missing Count", ascending=False)

print("--- TMDB Training Dataset Data Profile ---")
display(data_profile)

# Check for duplicate movie records
duplicate_count = train_df.duplicated(subset=["title", "release_date"]).sum()
print(f"\nDuplicate records based on title + release_date: {duplicate_count}")

# 4.2 Numerical Summary Statistics
print("\n--- Numerical Feature Summary Statistics ---")
display(train_df[["budget", "revenue", "runtime", "popularity"]].describe())

# 4.3 Inspect Key Metadata Columns
print("\nSample JSON-encoded columns in TMDB:")
sample_row = train_df.iloc[0]
for col in ["genres", "production_companies", "production_countries", "cast", "crew"]:
    print(f"  [{col}]: {str(sample_row[col])[:120]}...")

--- TMDB Training Dataset Data Profile ---


,Data Type,Non-Null Count,Missing Count,Missing Pct (%),Unique Values
belongs_to_collection,object,604,2396,79.87,422
homepage,object,946,2054,68.47,941
tagline,object,2403,597,19.90,2400
Keywords,object,2724,276,9.20,2648
production_companies,object,2844,156,5.20,2383
production_countries,object,2945,55,1.83,321
spoken_languages,object,2980,20,0.67,401
crew,object,2984,16,0.53,2984
cast,object,2987,13,0.43,2975
overview,object,2992,8,0.27,2992



Duplicate records based on title + release_date: 0

--- Numerical Feature Summary Statistics ---


,budget,revenue,runtime,popularity
count,3000.00,3000.00,2998.00,3000.00
mean,22531334.11,66725851.89,107.86,8.46
std,37026086.41,137532326.34,22.09,12.10
min,0.00,1.00,0.00,0.00
25%,0.00,2379808.25,94.00,4.02
50%,8000000.00,16807068.00,104.00,7.37
75%,29000000.00,68919203.50,118.00,10.89
max,380000000.00,1519557910.00,338.00,294.34



Sample JSON-encoded columns in TMDB:
  [genres]: [{'id': 35, 'name': 'Comedy'}]...
  [production_companies]: [{'name': 'Paramount Pictures', 'id': 4}, {'name': 'United Artists', 'id': 60}, {'name': 'Metro-Goldwyn-Mayer (MGM)', 'i...
  [production_countries]: [{'iso_3166_1': 'US', 'name': 'United States of America'}]...
  [cast]: [{'cast_id': 4, 'character': 'Lou', 'credit_id': '52fe4ee7c3a36847f82afae7', 'gender': 2, 'id': 52997, 'name': 'Rob Cord...
  [crew]: [{'credit_id': '59ac067c92514107af02c8c8', 'department': 'Directing', 'gender': 0, 'id': 1449071, 'job': 'First Assistan...


### Key Empirical Findings:
1. **Severe Zero-Budget Missingness:** 812 movies (27.1%) in the training set report `budget == 0`. In entertainment data collection, a budget of 0 is a sentinel indicating *unrecorded* expenditure, not a free movie.
2. **Extreme Target Skewness:** Box-office revenues span from \$1 to \$1.5 billion, with median revenue (\$16.8M) being far lower than mean revenue (\$66.7M), confirming an exponential distribution.
3. **Complex Nested JSON Metadata:** Key metadata columns (`genres`, `production_companies`, `production_countries`, `spoken_languages`, `cast`, `crew`) are stored as single-quoted stringified dictionaries and arrays requiring robust parsing.
4. **Missing Optional Fields:** `homepage` (68% missing) and `belongs_to_collection` (79% missing) are sparse, yet their mere presence indicates substantial studio backing and franchise status.

---
## Section 5: Data Cleaning & Parsing

We implement three foundational utilities directly inside the notebook:
1. `safe_parse_json()`: Safely unpacks JSON lists and dictionaries without crashing on single-quoted string representations, malformed syntax, or missing values.
2. `parse_release_date()`: Handles heterogeneous date formats (`MM/DD/YY`, `YYYY-MM-DD`), implementing a deterministic century pivot rule for historical two-digit years while strictly preserving explicit four-digit years.
3. `validate_dataset()`: Asserts that mandatory metadata columns exist in raw inputs.

In [4]:
def safe_parse_json(val: Any) -> List[Dict[str, Any]]:
    """
    Safely parses JSON-like string representations of dictionaries or lists.
    
    Handles:
    - Native Python lists, dicts, tuples (checks collection types first)
    - None, pd.NA, np.nan
    - Empty strings or empty bracket representations
    - Python literal strings using single quotes (ast.literal_eval)
    - Valid standard JSON strings (json.loads)
    - Malformed syntax (returns empty list without raising exceptions)
    """
    # 1. Native collections
    if isinstance(val, list):
        return [item for item in val if isinstance(item, dict)]
    if isinstance(val, tuple):
        return [item for item in val if isinstance(item, dict)]
    if isinstance(val, dict):
        return [val]
        
    # 2. Nulls and missing scalars
    if val is None or val is pd.NA:
        return []
    if isinstance(val, (float, np.floating)) and np.isnan(val):
        return []
        
    # 3. String validation
    if not isinstance(val, str):
        return []
        
    val_str = val.strip()
    if not val_str or val_str in ("[]", "{}", "null", "None", "nan", "NaN"):
        return []
        
    # 4. Python literal evaluation (handles single quotes in TMDB)
    try:
        parsed = ast.literal_eval(val_str)
        if isinstance(parsed, list):
            return [x for x in parsed if isinstance(x, dict)]
        elif isinstance(parsed, dict):
            return [parsed]
        return []
    except (ValueError, SyntaxError, MemoryError):
        pass
        
    # 5. Standard JSON fallback
    try:
        parsed = json.loads(val_str)
        if isinstance(parsed, list):
            return [x for x in parsed if isinstance(x, dict)]
        elif isinstance(parsed, dict):
            return [parsed]
        return []
    except (json.JSONDecodeError, ValueError):
        pass
        
    return []

def parse_release_date(series: pd.Series) -> pd.Series:
    """
    Parses release dates in formats (MM/DD/YY, MM/DD/YYYY, YYYY-MM-DD).
    
    Pivot Rule for 2-Digit Years:
    - If a 2-digit year produces a year > 2020 (e.g. '4/7/65' -> 2065), 100 years are subtracted
      to correctly map to 1965.
    - Explicit 4-digit years (e.g. '2025-07-18') are strictly preserved without subtraction.
    """
    if series is None or series.empty:
        return pd.Series(dtype="datetime64[ns]")
        
    def _parse_single(val: Any) -> Any:
        if val is None or val is pd.NA:
            return pd.NaT
        if isinstance(val, (float, np.floating)) and np.isnan(val):
            return pd.NaT
        if isinstance(val, (pd.Timestamp, np.datetime64)):
            return pd.to_datetime(val)
        if hasattr(val, "year") and hasattr(val, "month") and hasattr(val, "day"):
            return pd.to_datetime(val)
            
        val_str = str(val).strip()
        if not val_str or val_str.lower() in ("nan", "none", "nat", ""):
            return pd.NaT
            
        # If an explicit 4-digit year is present, parse directly
        if re.search(r"\b(18|19|20)\d{2}\b", val_str):
            return pd.to_datetime(val_str, errors="coerce")
            
        # Check for 2-digit year pattern: M/D/YY or MM/DD/YY
        parts = val_str.split("/")
        if len(parts) == 3 and len(parts[2]) == 2:
            dt = pd.to_datetime(val_str, format="%m/%d/%y", errors="coerce")
            if pd.notna(dt) and dt.year > 2020:
                try:
                    return dt.replace(year=dt.year - 100)
                except Exception:
                    return dt
            return dt
            
        return pd.to_datetime(val_str, errors="coerce")
        
    return series.apply(_parse_single)

def validate_dataset(df: pd.DataFrame, is_train: bool = True) -> None:
    """
    Validates that essential raw columns are present before proceeding.
    """
    mandatory = ["budget", "genres", "release_date", "runtime"]
    if is_train:
        mandatory.append("revenue")
    missing = [c for c in mandatory if c not in df.columns]
    if missing:
        raise ValueError(f"Input DataFrame is missing mandatory raw columns: {missing}")

# Test the cleaning functions on edge cases
test_json = "[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]"
parsed_test = safe_parse_json(test_json)
print(f"Safe JSON parsing test: {parsed_test}")

test_dates = pd.Series(["5/12/85", "11/20/19", "2024-06-15", "invalid_date", np.nan])
parsed_dates = parse_release_date(test_dates)
print("\nRelease date parsing test:")
for orig, dt in zip(test_dates, parsed_dates):
    print(f"  Original: '{orig}' -> Parsed: {dt}")

validate_dataset(train_df, is_train=True)
validate_dataset(test_df, is_train=False)
print("\nDataset schema validation passed for both train and test.")

Safe JSON parsing test: [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]

Release date parsing test:
  Original: '5/12/85' -> Parsed: 1985-05-12 00:00:00
  Original: '11/20/19' -> Parsed: 2019-11-20 00:00:00
  Original: '2024-06-15' -> Parsed: 2024-06-15 00:00:00
  Original: 'invalid_date' -> Parsed: NaT
  Original: 'nan' -> Parsed: NaT

Dataset schema validation passed for both train and test.


---
## Section 6: Exploratory Data Analysis (EDA)

We explore the statistical distributions, correlations, and business dynamics of worldwide box office revenue through 10 targeted visualizations.

In [5]:
# Filter labeled positive-revenue movies for EDA exploration
eda_df = train_df[train_df["revenue"] > 0].copy()
eda_df["parsed_release_date"] = parse_release_date(eda_df["release_date"])
eda_df["release_year"] = eda_df["parsed_release_date"].dt.year
eda_df["release_month"] = eda_df["parsed_release_date"].dt.month
eda_df["log_revenue"] = np.log1p(eda_df["revenue"])
eda_df["log_budget"] = np.log1p(eda_df["budget"])

# Extract primary genre and primary country for EDA
def extract_first_name(raw_val: Any) -> str:
    parsed = safe_parse_json(raw_val)
    if parsed and isinstance(parsed[0], dict) and "name" in parsed[0]:
        return str(parsed[0]["name"]).strip()
    return "Unknown"

def extract_first_country(raw_val: Any) -> str:
    parsed = safe_parse_json(raw_val)
    if parsed and isinstance(parsed[0], dict) and "iso_3166_1" in parsed[0]:
        return str(parsed[0]["iso_3166_1"]).strip().upper()
    return "Other"

eda_df["primary_genre"] = eda_df["genres"].apply(extract_first_name)
eda_df["primary_country"] = eda_df["production_countries"].apply(extract_first_country)
print(f"EDA subset prepared: {len(eda_df):,} movies with positive worldwide revenue.")

EDA subset prepared: 3,000 movies with positive worldwide revenue.


### Chart 1 & Chart 2: Worldwide Revenue Distribution (Raw vs. Log-Transformed)
*Analytical Takeaway:* Raw revenue is heavily right-skewed with extreme billion-dollar outliers; log-transformation normalizes the variance and stabilizes regression objectives.

In [6]:
fig1 = px.histogram(
    eda_df, x="revenue", nbins=60,
    title="Chart 1: Worldwide Revenue Distribution (Raw USD)",
    labels={"revenue": "Worldwide Revenue ($ USD)"},
    color_discrete_sequence=["#1f77b4"]
)
fig1.update_layout(height=400, yaxis_title="Movie Count")
fig1.show()

fig2 = px.histogram(
    eda_df, x="log_revenue", nbins=50,
    title="Chart 2: Log1p-Transformed Worldwide Revenue Distribution",
    labels={"log_revenue": "log(1 + Revenue)"},
    color_discrete_sequence=["#2ca02c"]
)
fig2.update_layout(height=400, yaxis_title="Movie Count")
fig2.show()

### Chart 3 & Chart 4: Production Budget Distribution & Budget vs. Revenue Correlation
*Analytical Takeaway:* Production budget exhibits a strong, positive log-linear correlation with worldwide box office ($r > 0.75$), serving as the single most powerful predictive signal.

In [7]:
# Positive budget distribution
pos_budget = eda_df[eda_df["budget"] > 0]

fig3 = px.histogram(
    pos_budget, x="budget", nbins=50,
    title="Chart 3: Production Budget Distribution (Non-Zero Budgets)",
    labels={"budget": "Production Budget ($ USD)"},
    color_discrete_sequence=["#ff7f0e"]
)
fig3.update_layout(height=400, yaxis_title="Movie Count")
fig3.show()

fig4 = px.scatter(
    pos_budget, x="budget", y="revenue",
    log_x=True, log_y=True,
    title="Chart 4: Budget vs. Revenue (Log-Log Scale with Trendline)",
    labels={"budget": "Production Budget ($ USD, Log Scale)", "revenue": "Worldwide Revenue ($ USD, Log Scale)"},
    opacity=0.65, color_discrete_sequence=["#1f77b4"]
)
# Add log-log fit line using standard numpy
log_b = np.log10(pos_budget["budget"])
log_r = np.log10(pos_budget["revenue"])
poly_m, poly_c = np.polyfit(log_b, log_r, 1)
x_vals = np.linspace(log_b.min(), log_b.max(), 50)
fig4.add_trace(go.Scatter(
    x=10**x_vals, y=10**(poly_m * x_vals + poly_c),
    mode="lines", name=f"Fit (slope={poly_m:.2f})", line=dict(color="red", width=2)
))
fig4.update_layout(height=450)
fig4.show()

### Chart 5 & Chart 6: Median Revenue by Genre & Historical Trend Across Release Decades
*Analytical Takeaway:* High-concept genres like Animation, Adventure, and Fantasy achieve the highest median box office, and worldwide revenues have grown substantially over recent decades.

In [8]:
# Median revenue by genre (minimum 15 movies)
genre_stats = eda_df.groupby("primary_genre")["revenue"].agg(["count", "median"]).reset_index()
genre_stats = genre_stats[genre_stats["count"] >= 15].sort_values(by="median", ascending=True)

fig5 = px.bar(
    genre_stats, x="median", y="primary_genre", orientation="h",
    title="Chart 5: Median Worldwide Revenue by Primary Genre (Min 15 Movies)",
    labels={"median": "Median Revenue ($ USD)", "primary_genre": "Genre"},
    color="median", color_continuous_scale="Viridis"
)
fig5.update_layout(height=480, showlegend=False)
fig5.show()

# Revenue trend across release years (1980 to 2018)
yearly_trend = eda_df[(eda_df["release_year"] >= 1980) & (eda_df["release_year"] <= 2018)]
yearly_stats = yearly_trend.groupby("release_year")["revenue"].agg(["mean", "median"]).reset_index()

fig6 = go.Figure()
fig6.add_trace(go.Scatter(x=yearly_stats["release_year"], y=yearly_stats["mean"], mode="lines+markers", name="Mean Revenue"))
fig6.add_trace(go.Scatter(x=yearly_stats["release_year"], y=yearly_stats["median"], mode="lines+markers", name="Median Revenue"))
fig6.update_layout(
    title="Chart 6: Historical Worldwide Revenue Trend (1980 - 2018)",
    xaxis_title="Release Year", yaxis_title="Worldwide Revenue ($ USD)",
    height=420
)
fig6.show()

### Chart 7 & Chart 8: Seasonal Release Timing & Production Country Dominance
*Analytical Takeaway:* Summer (May–July) and holiday windows (November–December) exhibit strong box-office spikes, while US-produced movies dominate global theatrical gross.

In [9]:
# Revenue by release month
monthly_stats = eda_df.groupby("release_month")["revenue"].median().reset_index()
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly_stats["month_name"] = monthly_stats["release_month"].apply(lambda m: month_labels[int(m)-1] if pd.notna(m) and 1 <= m <= 12 else "Unknown")

fig7 = px.bar(
    monthly_stats, x="month_name", y="revenue",
    title="Chart 7: Median Worldwide Revenue by Release Month",
    labels={"month_name": "Release Month", "revenue": "Median Revenue ($ USD)"},
    color_discrete_sequence=["#e377c2"]
)
fig7.update_layout(height=400)
fig7.show()

# Top 10 production countries by movie count
top_countries = eda_df.groupby("primary_country")["revenue"].agg(["count", "median"]).reset_index()
top_countries = top_countries[top_countries["count"] >= 10].sort_values(by="median", ascending=True)

fig8 = px.bar(
    top_countries, x="median", y="primary_country", orientation="h",
    title="Chart 8: Median Worldwide Revenue by Primary Production Country",
    labels={"median": "Median Revenue ($ USD)", "primary_country": "Country ISO Code"},
    color="median", color_continuous_scale="Blues"
)
fig8.update_layout(height=420, showlegend=False)
fig8.show()

### Chart 9 & Chart 10: Runtime Distribution & Runtime vs. Revenue
*Analytical Takeaway:* The majority of commercial films cluster between 90 and 125 minutes, with peak box-office returns observed in the 105–140 minute theatrical sweet spot.

In [10]:
valid_runtimes = eda_df[(eda_df["runtime"] >= 45) & (eda_df["runtime"] <= 240)]

fig9 = px.histogram(
    valid_runtimes, x="runtime", nbins=40,
    title="Chart 9: Theatrical Runtime Distribution (Minutes)",
    labels={"runtime": "Runtime (Minutes)"},
    color_discrete_sequence=["#8c564b"]
)
fig9.update_layout(height=400, yaxis_title="Movie Count")
fig9.show()

fig10 = px.scatter(
    valid_runtimes, x="runtime", y="revenue", log_y=True,
    title="Chart 10: Theatrical Runtime vs. Worldwide Revenue (Log Scale)",
    labels={"runtime": "Runtime (Minutes)", "revenue": "Worldwide Revenue ($ USD, Log Scale)"},
    opacity=0.6, color_discrete_sequence=["#9467bd"]
)
fig10.update_layout(height=430)
fig10.show()

---
## Section 7: Feature Engineering — Implement Everything Here

Feature engineering converts raw, heterogeneous metadata into a structured 37-dimensional pre-release feature representation:

### Canonical 37 Features:
1. **Numerical & Binary Metadata (21):**
   * `budget`: Imputed production budget in USD.
   * `log_budget`: $\log(1 + \text{budget})$.
   * `budget_is_zero`: Binary indicator ($1$ if budget was unrecorded/zero, else $0$).
   * `runtime`: Movie runtime in minutes (imputed with training median if missing).
   * `has_collection`: $1$ if part of a movie franchise/collection, else $0$.
   * `release_year`: Extracted calendar year.
   * `release_month`: Calendar month ($1$ to $12$).
   * `release_quarter`: Financial quarter ($1$ to $4$).
   * `release_dayofweek`: Day of week ($0 = \text{Monday}, 6 = \text{Sunday}$).
   * `release_is_weekend`: $1$ for Friday, Saturday, Sunday releases, else $0$.
   * `genre_count`: Total number of genres attached to the title.
   * `company_count`: Total number of production companies.
   * `is_major_studio`: $1$ if produced by a major Hollywood studio (Warner Bros, Disney, Universal, etc.), else $0$.
   * `country_count`: Number of producing countries.
   * `is_us_country`: $1$ if primary production country is the United States, else $0$.
   * `language_count`: Number of spoken languages.
   * `is_english`: $1$ if primary language is English, else $0$.
   * `cast_size`: Number of credited cast members.
   * `crew_size`: Number of credited crew members.
   * `has_homepage`: $1$ if promotional website exists, else $0$.
   * `has_tagline`: $1$ if promotional marketing tagline exists, else $0$.
2. **Categorical Attributes (2):**
   * `original_language`: ISO language code (top 12 retained, remainder mapped to `'other'`).
   * `primary_genre`: Lead genre name or `'None'`.
3. **Explicit Binary Genre Indicators (14):**
   * `genre_drama`, `genre_comedy`, `genre_thriller`, `genre_action`, `genre_romance`,
     `genre_crime`, `genre_adventure`, `genre_horror`, `genre_scifi`, `genre_family`,
     `genre_fantasy`, `genre_animation`, `genre_mystery`, `genre_documentary`.

### Zero Data Leakage Protocol:
* Imputation statistics (median budget, median runtime, median dates) are calculated **strictly on training data** and frozen into a `stats` dictionary.
* Validation and test sets **never compute their own imputation statistics**; they strictly reuse `stats`.

In [11]:
# Constants for Feature Engineering
MAJOR_STUDIOS = [
    "warner bros", "universal pictures", "paramount", "twentieth century fox",
    "20th century fox", "columbia pictures", "walt disney", "marvel",
    "sony pictures", "new line cinema", "metro-goldwyn-mayer", "mgm",
    "tri-star", "dreamworks", "lionsgate"
]

CORE_GENRES = [
    ("genre_drama", "Drama"),
    ("genre_comedy", "Comedy"),
    ("genre_thriller", "Thriller"),
    ("genre_action", "Action"),
    ("genre_romance", "Romance"),
    ("genre_crime", "Crime"),
    ("genre_adventure", "Adventure"),
    ("genre_horror", "Horror"),
    ("genre_scifi", "Science Fiction"),
    ("genre_family", "Family"),
    ("genre_fantasy", "Fantasy"),
    ("genre_animation", "Animation"),
    ("genre_mystery", "Mystery"),
    ("genre_documentary", "Documentary"),
]

TOP_LANGUAGES = {"en", "fr", "es", "zh", "de", "hi", "ja", "it", "ko", "ru", "cn", "pt"}

BASE_FEATURE_COLUMNS = [
    "budget", "log_budget", "budget_is_zero", "runtime", "has_collection",
    "release_year", "release_month", "release_quarter", "release_dayofweek",
    "release_is_weekend", "genre_count", "company_count", "is_major_studio",
    "country_count", "is_us_country", "language_count", "is_english",
    "cast_size", "crew_size", "has_homepage", "has_tagline",
    "original_language", "primary_genre"
]

ALL_FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + [col_name for col_name, _ in CORE_GENRES]
assert len(ALL_FEATURE_COLUMNS) == 37, f"Expected 37 features, got {len(ALL_FEATURE_COLUMNS)}"

def compute_training_stats(data: pd.DataFrame) -> Dict[str, Any]:
    """
    Computes reference imputation statistics solely from training records.
    Validation and test records must NEVER influence these statistics.
    """
    non_zero_budgets = pd.to_numeric(data["budget"], errors="coerce")
    non_zero_budgets = non_zero_budgets[non_zero_budgets > 0]
    median_budget = float(non_zero_budgets.median()) if len(non_zero_budgets) > 0 else 16000000.0
    
    valid_runtimes = pd.to_numeric(data["runtime"], errors="coerce")
    valid_runtimes = valid_runtimes[valid_runtimes > 0]
    median_runtime = float(valid_runtimes.median()) if len(valid_runtimes) > 0 else 104.0
    
    parsed_dates = parse_release_date(data["release_date"])
    valid_years = parsed_dates.dt.year.dropna()
    valid_months = parsed_dates.dt.month.dropna()
    valid_dows = parsed_dates.dt.dayofweek.dropna()
    
    median_year = int(valid_years.median()) if len(valid_years) > 0 else 2004
    median_month = int(valid_months.median()) if len(valid_months) > 0 else 6
    median_dow = int(valid_dows.median()) if len(valid_dows) > 0 else 4
    
    return {
        "median_budget": median_budget,
        "median_runtime": median_runtime,
        "budget_p99": float(non_zero_budgets.quantile(0.99)) if len(non_zero_budgets) > 0 else 200000000.0,
        "runtime_min": float(valid_runtimes.min()) if len(valid_runtimes) > 0 else 40.0,
        "runtime_max": float(valid_runtimes.max()) if len(valid_runtimes) > 0 else 240.0,
        "median_release_year": median_year,
        "median_release_month": median_month,
        "median_release_dayofweek": median_dow,
    }

def engineer_movie_features(
    df: pd.DataFrame,
    is_train: bool = True,
    stats: Optional[Dict[str, Any]] = None
) -> Tuple[pd.DataFrame, Optional[pd.Series], Dict[str, Any]]:
    """
    Transforms raw TMDB metadata into 37 canonical pre-release predictive features.
    """
    validate_dataset(df, is_train=False)
    data = df.copy()
    
    # 1. Target extraction
    y: Optional[pd.Series] = None
    if "revenue" in data.columns and is_train:
        y_raw = pd.to_numeric(data["revenue"], errors="coerce")
        valid_mask = y_raw.notna() & (y_raw > 0)
        data = data.loc[valid_mask].copy()
        y = y_raw.loc[valid_mask]
        
    # 2. Imputation statistics
    if stats is None:
        stats = compute_training_stats(data)
        
    median_budget = stats["median_budget"]
    median_runtime = stats["median_runtime"]
    median_year = stats.get("median_release_year", 2004)
    median_month = stats.get("median_release_month", 6)
    median_dow = stats.get("median_release_dayofweek", 4)
    
    # 3. Budget features
    budget_numeric = pd.to_numeric(data["budget"], errors="coerce").fillna(0)
    budget_is_zero = (budget_numeric <= 0).astype(int)
    budget_imputed = np.where(budget_numeric > 0, budget_numeric, median_budget)
    log_budget = np.log1p(budget_imputed)
    
    # 4. Runtime features
    runtime_numeric = pd.to_numeric(data["runtime"], errors="coerce")
    runtime_imputed = np.where(runtime_numeric.notna() & (runtime_numeric > 0), runtime_numeric, median_runtime)
    
    # 5. Collection flag
    if "belongs_to_collection" in data.columns:
        has_collection = (
            data["belongs_to_collection"].notna()
            & (data["belongs_to_collection"].astype(str).str.strip() != "")
            & (data["belongs_to_collection"].astype(str).str.strip() != "[]")
            & (data["belongs_to_collection"].astype(str).str.strip() != "{}")
        ).astype(int)
    else:
        has_collection = pd.Series(0, index=data.index, dtype=int)
        
    # 6. Release date features
    parsed_dates = parse_release_date(data["release_date"])
    release_year = parsed_dates.dt.year.fillna(median_year).astype(int)
    release_month = parsed_dates.dt.month.fillna(median_month).astype(int)
    release_quarter = ((release_month - 1) // 3 + 1).astype(int)
    release_dayofweek = parsed_dates.dt.dayofweek.fillna(median_dow).astype(int)
    release_is_weekend = release_dayofweek.isin([4, 5, 6]).astype(int)
    
    # 7. Genres parsing
    parsed_genres = data["genres"].apply(safe_parse_json)
    genre_count = parsed_genres.apply(len).astype(int)
    
    def get_primary_genre(g_list: List[Dict[str, Any]]) -> str:
        if g_list and isinstance(g_list[0], dict) and "name" in g_list[0]:
            name = str(g_list[0]["name"]).strip()
            return name if name else "None"
        return "None"
        
    primary_genre = parsed_genres.apply(get_primary_genre)
    
    genre_flags: Dict[str, pd.Series] = {}
    for col_name, g_target in CORE_GENRES:
        genre_flags[col_name] = parsed_genres.apply(
            lambda glist: int(any(isinstance(g, dict) and g.get("name") == g_target for g in glist))
        )
        
    # 8. Companies parsing
    if "production_companies" in data.columns:
        parsed_companies = data["production_companies"].apply(safe_parse_json)
        company_count = parsed_companies.apply(len).astype(int)
        
        def check_major_studio(c_list: List[Dict[str, Any]]) -> int:
            for c in c_list:
                if isinstance(c, dict):
                    c_name = str(c.get("name", "")).lower()
                    if any(studio in c_name for studio in MAJOR_STUDIOS):
                        return 1
            return 0
            
        is_major_studio = parsed_companies.apply(check_major_studio).astype(int)
    else:
        company_count = pd.Series(0, index=data.index, dtype=int)
        is_major_studio = pd.Series(0, index=data.index, dtype=int)
        
    # 9. Countries parsing
    if "production_countries" in data.columns:
        parsed_countries = data["production_countries"].apply(safe_parse_json)
        country_count = parsed_countries.apply(len).astype(int)
        
        def get_primary_country(c_list: List[Dict[str, Any]]) -> str:
            if c_list and isinstance(c_list[0], dict) and "iso_3166_1" in c_list[0]:
                code = str(c_list[0]["iso_3166_1"]).strip().upper()
                return code if code else "OTHER"
            return "OTHER"
            
        primary_country = parsed_countries.apply(get_primary_country)
        is_us_country = (primary_country == "US").astype(int)
    else:
        country_count = pd.Series(0, index=data.index, dtype=int)
        is_us_country = pd.Series(0, index=data.index, dtype=int)
        
    # 10. Spoken languages
    if "spoken_languages" in data.columns:
        parsed_languages = data["spoken_languages"].apply(safe_parse_json)
        language_count = parsed_languages.apply(len).astype(int)
    else:
        language_count = pd.Series(0, index=data.index, dtype=int)
        
    # 11. Original language
    if "original_language" in data.columns:
        raw_lang = data["original_language"].fillna("other").astype(str).str.strip().str.lower()
        clean_lang = raw_lang.apply(lambda l: l if l in TOP_LANGUAGES else "other")
        is_english = (clean_lang == "en").astype(int)
    else:
        clean_lang = pd.Series("other", index=data.index)
        is_english = pd.Series(0, index=data.index, dtype=int)
        
    # 12. Cast & crew sizes
    if "cast" in data.columns:
        parsed_cast = data["cast"].apply(safe_parse_json)
        cast_size = parsed_cast.apply(len).astype(int)
    else:
        cast_size = pd.Series(0, index=data.index, dtype=int)
        
    if "crew" in data.columns:
        parsed_crew = data["crew"].apply(safe_parse_json)
        crew_size = parsed_crew.apply(len).astype(int)
    else:
        crew_size = pd.Series(0, index=data.index, dtype=int)
        
    # 13. Pre-release marketing indicators
    if "homepage" in data.columns:
        has_homepage = (
            data["homepage"].notna()
            & (data["homepage"].astype(str).str.strip() != "")
            & (data["homepage"].astype(str).str.strip().str.lower() != "nan")
        ).astype(int)
    else:
        has_homepage = pd.Series(0, index=data.index, dtype=int)
        
    if "tagline" in data.columns:
        has_tagline = (
            data["tagline"].notna()
            & (data["tagline"].astype(str).str.strip() != "")
            & (data["tagline"].astype(str).str.strip().str.lower() != "nan")
        ).astype(int)
    else:
        has_tagline = pd.Series(0, index=data.index, dtype=int)
        
    # Build complete feature matrix
    features = pd.DataFrame({
        "budget": budget_imputed,
        "log_budget": log_budget,
        "budget_is_zero": budget_is_zero,
        "runtime": runtime_imputed,
        "has_collection": has_collection,
        "release_year": release_year,
        "release_month": release_month,
        "release_quarter": release_quarter,
        "release_dayofweek": release_dayofweek,
        "release_is_weekend": release_is_weekend,
        "genre_count": genre_count,
        "company_count": company_count,
        "is_major_studio": is_major_studio,
        "country_count": country_count,
        "is_us_country": is_us_country,
        "language_count": language_count,
        "is_english": is_english,
        "cast_size": cast_size,
        "crew_size": crew_size,
        "has_homepage": has_homepage,
        "has_tagline": has_tagline,
        "original_language": clean_lang,
        "primary_genre": primary_genre,
    }, index=data.index)
    
    for col_name, series in genre_flags.items():
        features[col_name] = series
        
    features = features[ALL_FEATURE_COLUMNS]
    return features, y, stats

def prepare_single_movie_features(
    movie_dict: Dict[str, Any],
    stats: Dict[str, Any]
) -> pd.DataFrame:
    """
    Transforms a single movie dictionary into a 1-row DataFrame matching the exact
    canonical 37 columns and transformation logic.
    """
    median_budget = stats.get("median_budget", 16000000.0)
    median_runtime = stats.get("median_runtime", 104.0)
    median_year = stats.get("median_release_year", 2004)
    median_month = stats.get("median_release_month", 6)
    median_dow = stats.get("median_release_dayofweek", 4)
    
    budget_raw = float(movie_dict.get("budget", 0.0) or 0.0)
    budget_is_zero = 1 if budget_raw <= 0 else 0
    budget_imputed = budget_raw if budget_raw > 0 else median_budget
    log_budget = float(np.log1p(budget_imputed))
    
    runtime_raw = float(movie_dict.get("runtime", 0.0) or 0.0)
    runtime_imputed = runtime_raw if runtime_raw > 0 else median_runtime
    
    has_collection = 1 if bool(movie_dict.get("belongs_to_collection", False)) else 0
    
    rel_date = movie_dict.get("release_date")
    if rel_date is not None and hasattr(rel_date, "year"):
        release_year = int(rel_date.year)
        release_month = int(rel_date.month)
        release_quarter = int((rel_date.month - 1) // 3 + 1)
        release_dayofweek = int(rel_date.weekday())
    elif rel_date is not None and isinstance(rel_date, str) and rel_date.strip():
        parsed = parse_release_date(pd.Series([rel_date]))
        if pd.notna(parsed.iloc[0]):
            dt = parsed.iloc[0]
            release_year = int(dt.year)
            release_month = int(dt.month)
            release_quarter = int((dt.month - 1) // 3 + 1)
            release_dayofweek = int(dt.dayofweek)
        else:
            release_year, release_month, release_quarter, release_dayofweek = median_year, median_month, (median_month-1)//3+1, median_dow
    else:
        release_year, release_month, release_quarter, release_dayofweek = median_year, median_month, (median_month-1)//3+1, median_dow
        
    release_is_weekend = 1 if release_dayofweek in (4, 5, 6) else 0
    
    raw_genres = movie_dict.get("genres", [])
    if isinstance(raw_genres, str):
        raw_genres = [g.strip() for g in raw_genres.split(",") if g.strip()]
    selected_genres = [str(g).strip() for g in raw_genres if str(g).strip()]
    genre_count = len(selected_genres)
    primary_genre = selected_genres[0] if selected_genres else "None"
    
    genre_flags: Dict[str, int] = {}
    for col_name, g_target in CORE_GENRES:
        genre_flags[col_name] = 1 if g_target in selected_genres else 0
        
    company_count = int(movie_dict.get("company_count", 1) or 1)
    is_major = 1 if bool(movie_dict.get("is_major_studio", False)) else 0
    
    primary_country = str(movie_dict.get("production_country", "US") or "OTHER").strip().upper()
    country_count = int(movie_dict.get("country_count", 1) or 1)
    is_us_country = 1 if primary_country == "US" else 0
    
    raw_lang = str(movie_dict.get("original_language", "en") or "other").strip().lower()
    clean_lang = raw_lang if raw_lang in TOP_LANGUAGES else "other"
    is_english = 1 if clean_lang == "en" else 0
    language_count = int(movie_dict.get("language_count", 1) or 1)
    
    cast_size = int(movie_dict.get("cast_size", 15) or 15)
    crew_size = int(movie_dict.get("crew_size", 20) or 20)
    
    has_homepage = 1 if bool(movie_dict.get("has_homepage", False)) else 0
    has_tagline = 1 if bool(movie_dict.get("has_tagline", False)) else 0
    
    row_data: Dict[str, Any] = {
        "budget": budget_imputed,
        "log_budget": log_budget,
        "budget_is_zero": budget_is_zero,
        "runtime": runtime_imputed,
        "has_collection": has_collection,
        "release_year": release_year,
        "release_month": release_month,
        "release_quarter": release_quarter,
        "release_dayofweek": release_dayofweek,
        "release_is_weekend": release_is_weekend,
        "genre_count": genre_count,
        "company_count": company_count,
        "is_major_studio": is_major,
        "country_count": country_count,
        "is_us_country": is_us_country,
        "language_count": language_count,
        "is_english": is_english,
        "cast_size": cast_size,
        "crew_size": crew_size,
        "has_homepage": has_homepage,
        "has_tagline": has_tagline,
        "original_language": clean_lang,
        "primary_genre": primary_genre,
    }
    for col_name, val in genre_flags.items():
        row_data[col_name] = val
        
    single_df = pd.DataFrame([row_data])
    return single_df[ALL_FEATURE_COLUMNS]

print("Feature engineering modules defined successfully.")
print(f"Total Canonical Features: {len(ALL_FEATURE_COLUMNS)}")

Feature engineering modules defined successfully.
Total Canonical Features: 37


---
## Section 8: Target Leakage Prevention

### Academic Rationale & Discussion:
Target leakage occurs when training features incorporate information that would not be accessible at the point of prediction in production.
1. **Target Variable (`revenue`):** Directly predicting revenue using revenue or any derivative is a trivial error.
2. **Post-Release Audience Signals (`popularity`, `vote_average`, `vote_count`):** These metrics are compiled *after* thousands of people have purchased tickets and rated the movie on TMDB. Using them transforms the problem from *revenue prediction* into *post-release popularity calibration*.
3. **Identifiers (`id`, `imdb_id`, `poster_path`):** Arbitrary database keys can act as accidental memorization shortcuts for tree ensembles.
4. **Questionable Metadata Timing (`belongs_to_collection`, `homepage`, `tagline`):** While franchise status, marketing taglines, and official domains are planned pre-release, historical database backfills can sometimes occur. We treat them as valid pre-release marketing signals while monitoring their tree importance.

In [12]:
# Demonstration of Leakage Prevention Check
LEAKAGE_COLUMNS = [
    "revenue", "popularity", "vote_average", "vote_count",
    "id", "imdb_id", "poster_path", "overview", "status"
]

def assert_zero_target_leakage(feature_df: pd.DataFrame) -> None:
    """
    Raises an AssertionError if any prohibited leakage column is present in features.
    """
    violators = [col for col in LEAKAGE_COLUMNS if col in feature_df.columns]
    assert len(violators) == 0, f"Critical: Target leakage detected! Prohibited columns present: {violators}"
    print(f"Target Leakage Audit PASSED: 0 prohibited columns found among {feature_df.shape[1]} engineered features.")

# Preliminary test on feature column list
assert_zero_target_leakage(pd.DataFrame(columns=ALL_FEATURE_COLUMNS))

Target Leakage Audit PASSED: 0 prohibited columns found among 37 engineered features.


---
## Section 9: Train and Validation Split

### Preventing Preprocessing Data Leakage:
A critical tenet of statistical machine learning: **the validation set must simulate completely unseen future data**.
If imputation statistics (such as median budget or median runtime) are calculated across the entire dataset before splitting, the validation set's distribution leaks into the training process.

### Workflow:
1. Filter training records with strictly valid, positive revenue labels ($y > 0$).
2. Split raw records 80/20 using `train_test_split(random_state=42)`.
3. Compute `training_stats` **solely from the 80% training partition**.
4. Transform training records with `engineer_movie_features(train_raw, stats=training_stats)`.
5. Transform validation records reusing `training_stats`.

In [13]:
# 1. Filter positive labeled training data
labeled_mask = train_df["revenue"].notna() & (train_df["revenue"] > 0)
clean_train_raw = train_df.loc[labeled_mask].copy().reset_index(drop=True)
print(f"Total labeled records with positive revenue: {len(clean_train_raw):,} / {len(train_df):,}")

# 2. Split raw records 80% train, 20% validation
train_raw_split, val_raw_split = train_test_split(
    clean_train_raw,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print(f"Raw Split: Training partition = {len(train_raw_split):,} movies | Validation partition = {len(val_raw_split):,} movies")

# 3. Derive imputation statistics STRICTLY on the training partition
partition_stats = compute_training_stats(train_raw_split)
print("\nDerived Training Partition Statistics (Validation data was NOT seen):")
for k, v in partition_stats.items():
    print(f"  {k}: {v}")

# 4. Transform training partition
X_train, y_train, _ = engineer_movie_features(train_raw_split, is_train=True, stats=partition_stats)

# 5. Transform validation partition reusing training partition stats
X_val, y_val, _ = engineer_movie_features(val_raw_split, is_train=True, stats=partition_stats)

# Audit target leakage on both splits
assert_zero_target_leakage(X_train)
assert_zero_target_leakage(X_val)

print(f"\nProcessed Training Feature Matrix:   X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Processed Validation Feature Matrix: X_val   {X_val.shape}, y_val   {y_val.shape}")

print("\nSample Engineered Feature Rows (X_train.head(3)):")
display(X_train.head(3))

Total labeled records with positive revenue: 3,000 / 3,000
Raw Split: Training partition = 2,400 movies | Validation partition = 600 movies



Derived Training Partition Statistics (Validation data was NOT seen):
  median_budget: 15000000.0
  median_runtime: 104.0
  budget_p99: 200000000.0
  runtime_min: 11.0
  runtime_max: 248.0
  median_release_year: 2004
  median_release_month: 7
  median_release_dayofweek: 4


Target Leakage Audit PASSED: 0 prohibited columns found among 37 engineered features.
Target Leakage Audit PASSED: 0 prohibited columns found among 37 engineered features.

Processed Training Feature Matrix:   X_train (2400, 37), y_train (2400,)
Processed Validation Feature Matrix: X_val   (600, 37), y_val   (600,)

Sample Engineered Feature Rows (X_train.head(3)):


,budget,log_budget,budget_is_zero,runtime,has_collection,release_year,release_month,release_quarter,release_dayofweek,release_is_weekend,genre_count,company_count,is_major_studio,country_count,is_us_country,language_count,is_english,cast_size,crew_size,has_homepage,has_tagline,original_language,primary_genre,genre_drama,genre_comedy,genre_thriller,genre_action,genre_romance,genre_crime,genre_adventure,genre_horror,genre_scifi,genre_family,genre_fantasy,genre_animation,genre_mystery,genre_documentary
642,39000000.00,17.48,0,116.00,0,1988,6,2,1,0,2,2,1,1,1,1,1,14,12,0,1,en,Comedy,0,1,0,0,1,0,0,0,0,0,0,0,0,0
700,25000000.00,17.03,0,94.00,0,1986,12,4,4,1,3,2,1,1,1,1,1,17,15,0,1,en,Horror,0,1,0,0,0,0,0,1,0,0,0,0,0,0
226,50000000.00,17.73,0,105.00,0,2011,2,1,3,0,4,5,0,1,1,1,1,51,70,1,1,en,Fantasy,0,0,1,1,0,1,0,0,0,0,1,0,0,0


---
## Section 10: Preprocessing Pipeline

We construct a Scikit-Learn `ColumnTransformer` that:
1. Imputes and scales numerical variables (`StandardScaler`).
2. Encodes categorical variables (`original_language`, `primary_genre`) using `OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=30)`.
3. Passes binary flags through without modification.

In [14]:
def build_preprocessor(feature_columns: List[str]) -> Tuple[ColumnTransformer, List[str], List[str]]:
    """
    Builds a ColumnTransformer that scales numerical variables and one-hot encodes
    low-cardinality categorical variables.
    """
    cat_cols = ["original_language", "primary_genre"]
    cat_cols = [c for c in cat_cols if c in feature_columns]
    num_cols = [c for c in feature_columns if c not in cat_cols]
    
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=30)),
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipeline, num_cols),
            ("cat", cat_pipeline, cat_cols),
        ],
        remainder="drop"
    )
    return preprocessor, num_cols, cat_cols

# Test pipeline construction
test_preprocessor, num_features, cat_features = build_preprocessor(ALL_FEATURE_COLUMNS)
print(f"Preprocessor configured: {len(num_features)} numerical/binary features, {len(cat_features)} categorical features.")

Preprocessor configured: 35 numerical/binary features, 2 categorical features.


---
## Section 11: Machine Learning Model Training

We train and compare four diverse regression architectures:
1. **Dummy Baseline Regressor:** Predicts the median training log-revenue. Provides a minimum sanity threshold.
2. **Random Forest Regressor:** Bagged ensemble of decorrelated decision trees (`n_estimators=100`, `max_depth=12`, `min_samples_split=4`).
3. **Extra Trees Regressor:** Extremely randomized trees that draw random split thresholds, reducing ensemble variance.
4. **HistGradientBoosting Regressor:** Modern histogram-based gradient boosting regressor (`max_iter=100`, `max_depth=6`, `learning_rate=0.08`).

### How Training Happens in the Code Cell:
1. The target is converted into log-space: `y_train_log = np.log1p(np.clip(y_train, 0, None))`.
2. The complete Scikit-Learn `Pipeline` is fitted: `pipeline.fit(X_train, y_train_log)`.
3. Validation predictions are computed: `y_val_log_pred = pipeline.predict(X_val)`.
4. Predictions are transformed back to USD: `y_val_pred = np.clip(np.expm1(y_val_log_pred), 0, None)`.
5. RMSLE, MAE, and $R^2$ validation metrics are calculated.

In [15]:
def compute_rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Computes Root Mean Squared Logarithmic Error (RMSLE).
    Ensures predictions and ground truth are non-negative.
    """
    y_t = np.asarray(y_true, dtype=float)
    y_p = np.asarray(y_pred, dtype=float)
    if len(y_t) == 0 or len(y_p) == 0:
        return 0.0
    clipped_pred = np.clip(y_p, 0, None)
    clipped_true = np.clip(y_t, 0, None)
    return float(np.sqrt(np.mean((np.log1p(clipped_pred) - np.log1p(clipped_true)) ** 2)))

def create_model(model_name: str, hyperparams: Optional[Dict[str, Any]] = None, random_state: int = 42) -> Any:
    """
    Instantiates the selected regression estimator.
    """
    params = hyperparams or {}
    if model_name == "dummy":
        return DummyRegressor(strategy="median")
    elif model_name == "random_forest":
        return RandomForestRegressor(
            n_estimators=params.get("n_estimators", 100),
            max_depth=params.get("max_depth", 12),
            min_samples_split=params.get("min_samples_split", 4),
            random_state=random_state,
            n_jobs=-1
        )
    elif model_name == "extra_trees":
        return ExtraTreesRegressor(
            n_estimators=params.get("n_estimators", 100),
            max_depth=params.get("max_depth", 12),
            min_samples_split=params.get("min_samples_split", 4),
            random_state=random_state,
            n_jobs=-1
        )
    elif model_name == "hist_gradient_boosting":
        return HistGradientBoostingRegressor(
            max_iter=params.get("max_iter", 100),
            max_depth=params.get("max_depth", 6),
            learning_rate=params.get("learning_rate", 0.08),
            random_state=random_state
        )
    else:
        raise ValueError(f"Unknown model name '{model_name}'.")

def train_and_evaluate_model(
    model_name: str,
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    X_v: pd.DataFrame,
    y_v: pd.Series,
    hyperparams: Optional[Dict[str, Any]] = None,
    random_state: int = 42
) -> Dict[str, Any]:
    """
    Fits full preprocessing + estimator pipeline on log1p(y_tr), evaluates against y_v.
    """
    preprocessor, num_cols, cat_cols = build_preprocessor(list(X_tr.columns))
    estimator = create_model(model_name, hyperparams, random_state=random_state)
    
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("estimator", estimator)
    ])
    
    # 1. Transform training target to log1p space
    y_tr_log = np.log1p(np.clip(y_tr.values, 0, None))
    
    # 2. FIT THE PIPELINE
    t0 = time.time()
    pipeline.fit(X_tr, y_tr_log)
    fit_duration = round(time.time() - t0, 3)
    
    # 3. Predict on validation set
    y_val_log_pred = pipeline.predict(X_v)
    
    # 4. Transform back to dollar revenue
    y_val_pred = np.clip(np.expm1(y_val_log_pred), 0, None)
    y_val_true = y_v.values
    
    # 5. Compute validation metrics
    rmsle = compute_rmsle(y_val_true, y_val_pred)
    mae = float(mean_absolute_error(y_val_true, y_val_pred))
    r2 = float(r2_score(y_val_true, y_val_pred))
    
    # Extract feature importances if available
    df_importances = pd.DataFrame(columns=["feature", "importance"])
    if hasattr(estimator, "feature_importances_"):
        importances = estimator.feature_importances_
        cat_names = []
        try:
            cat_pipe = pipeline.named_steps["preprocessor"].named_transformers_["cat"]
            cat_names = list(cat_pipe.named_steps["onehot"].get_feature_names_out(cat_cols))
        except Exception:
            cat_names = [f"cat_{i}" for i in range(len(importances) - len(num_cols))]
        all_feat_names = num_cols + cat_names
        if len(all_feat_names) == len(importances):
            df_importances = pd.DataFrame({
                "feature": all_feat_names,
                "importance": importances
            }).sort_values(by="importance", ascending=False).reset_index(drop=True)
            
    return {
        "model_name": model_name,
        "pipeline": pipeline,
        "duration_sec": fit_duration,
        "rmsle": rmsle,
        "mae": mae,
        "r2": r2,
        "y_val_true": y_val_true,
        "y_val_pred": y_val_pred,
        "residuals": y_val_true - y_val_pred,
        "feature_importances": df_importances,
    }

# Execute Training Across All 4 Models
models_to_train = [
    ("dummy", {}),
    ("random_forest", {"n_estimators": 100, "max_depth": 12, "min_samples_split": 4}),
    ("extra_trees", {"n_estimators": 100, "max_depth": 12, "min_samples_split": 4}),
    ("hist_gradient_boosting", {"max_iter": 100, "max_depth": 6, "learning_rate": 0.08}),
]

training_results = {}
for m_name, hparams in models_to_train:
    print(f"Training {m_name}...")
    res = train_and_evaluate_model(m_name, X_train, y_train, X_val, y_val, hyperparams=hparams, random_state=RANDOM_STATE)
    training_results[m_name] = res
    print(f"  -> Validation RMSLE: {res['rmsle']:.4f} | MAE: ${res['mae']:,.0f} | R²: {res['r2']:.4f} ({res['duration_sec']}s)")

print("\nModel training across all 4 architectures successfully completed.")

Training dummy...
  -> Validation RMSLE: 3.0203 | MAE: $66,076,091 | R²: -0.1811 (0.044s)
Training random_forest...


  -> Validation RMSLE: 2.0241 | MAE: $37,769,439 | R²: 0.6632 (0.521s)
Training extra_trees...


  -> Validation RMSLE: 1.9926 | MAE: $37,917,548 | R²: 0.6669 (0.429s)
Training hist_gradient_boosting...


  -> Validation RMSLE: 2.0592 | MAE: $38,997,927 | R²: 0.6501 (3.919s)

Model training across all 4 architectures successfully completed.


---
## Section 12: Model Comparison & Champion Selection

We assemble the validation metrics into a comparison leaderboard and systematically select the champion model (lowest RMSLE).

In [16]:
comparison_rows = []
for m_name, res in training_results.items():
    comparison_rows.append({
        "Model Key": m_name,
        "Validation RMSLE (↓)": round(res["rmsle"], 4),
        "Validation MAE ($ USD)": f"${res['mae']:,.0f}",
        "Validation R² (↑)": round(res["r2"], 4),
        "Train Time (s)": res["duration_sec"]
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(by="Validation RMSLE (↓)").reset_index(drop=True)
print("=== Model Performance Comparison Leaderboard ===")
display(comparison_df)

# Identify Champion Model
champion_key = comparison_df.iloc[0]["Model Key"]
champion_res = training_results[champion_key]
print(f"\nCHAMPION MODEL SELECTED: '{champion_key.upper()}' with lowest Validation RMSLE = {champion_res['rmsle']:.4f}")

=== Model Performance Comparison Leaderboard ===


,Model Key,Validation RMSLE (↓),Validation MAE ($ USD),Validation R² (↑),Train Time (s)
0,extra_trees,1.99,"$37,917,548",0.67,0.43
1,random_forest,2.02,"$37,769,439",0.66,0.52
2,hist_gradient_boosting,2.06,"$38,997,927",0.65,3.92
3,dummy,3.02,"$66,076,091",-0.18,0.04



CHAMPION MODEL SELECTED: 'EXTRA_TREES' with lowest Validation RMSLE = 1.9926


---
## Section 13: Model Evaluation (Champion Model Diagnostics)

For the champion model, we generate:
1. **Actual vs. Predicted Scatter Plot:** With a $45^\circ$ ($y = x$) reference line.
2. **Residual Analysis:** Distribution of prediction errors ($y - \hat{y}$).
3. **Feature Importance Ranking:** Top predictive drivers identified by tree splits.

In [17]:
# 13.1 Actual vs. Predicted Plot (Log Scale)
eval_df = pd.DataFrame({
    "actual": champion_res["y_val_true"],
    "predicted": champion_res["y_val_pred"]
})

fig_eval = px.scatter(
    eval_df, x="actual", y="predicted", log_x=True, log_y=True,
    title=f"Chart 11: Actual vs. Predicted Worldwide Revenue ({champion_key.title()})",
    labels={"actual": "Actual Revenue ($ USD, Log Scale)", "predicted": "Predicted Revenue ($ USD, Log Scale)"},
    opacity=0.65, color_discrete_sequence=["#1f77b4"]
)

# Add 45-degree reference line
min_val = min(eval_df["actual"].min(), eval_df["predicted"].min())
max_val = max(eval_df["actual"].max(), eval_df["predicted"].max())
fig_eval.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode="lines", name="Ideal y = x", line=dict(color="red", dash="dash")
))
fig_eval.update_layout(height=480)
fig_eval.show()

# 13.2 Residual Distribution Plot
residuals = champion_res["residuals"]
# Trim extreme outliers solely for visualization clarity
q01, q99 = np.percentile(residuals, [1, 99])
trimmed_residuals = residuals[(residuals >= q01) & (residuals <= q99)]

fig_res = px.histogram(
    trimmed_residuals, nbins=50,
    title=f"Chart 12: Residual Distribution [y_actual - y_pred] ({champion_key.title()})",
    labels={"value": "Residual ($ USD, 1st–99th Percentile)"},
    color_discrete_sequence=["#d62728"]
)
fig_res.add_vline(x=0, line_dash="dash", line_color="black", annotation_text="Zero Error")
fig_res.update_layout(height=400, yaxis_title="Validation Movie Count")
fig_res.show()

# 13.3 Feature Importance Ranking (if available)
imp_df = champion_res["feature_importances"]
if not imp_df.empty:
    top_imp = imp_df.head(15).sort_values(by="importance", ascending=True)
    fig_imp = px.bar(
        top_imp, x="importance", y="feature", orientation="h",
        title=f"Chart 13: Top 15 Feature Importances ({champion_key.title()})",
        labels={"importance": "Relative Importance", "feature": "Feature Name"},
        color="importance", color_continuous_scale="Purples"
    )
    fig_imp.update_layout(height=480, showlegend=False)
    fig_imp.show()
else:
    print(f"Feature importance plot skipped: {champion_key} does not provide direct tree importances.")

---
## Section 14: Interactive Single-Movie Prediction

We implement `predict_movie_revenue()` directly in the notebook and test it on two realistic film scenarios:
1. **Scenario A (Mid-Budget Studio Action-Thriller):** \$45M budget, major studio backing, star cast.
2. **Scenario B (Zero-Budget Independent Film):** Demonstrates graceful handling of unrecorded budget using median training imputation.

In [18]:
def predict_movie_revenue(pipeline: Pipeline, single_row_df: pd.DataFrame) -> float:
    """
    Predicts worldwide box office revenue in USD for a single movie feature row.
    Guarantees non-negative prediction.
    """
    pred_log = pipeline.predict(single_row_df)[0]
    return float(np.clip(np.expm1(pred_log), 0, None))

# Test Scenario A: Mid-Budget Studio Action Film
movie_input_A = {
    "budget": 45000000,
    "runtime": 118,
    "release_date": "2027-07-18",
    "genres": ["Action", "Thriller"],
    "original_language": "en",
    "production_country": "US",
    "company_count": 2,
    "is_major_studio": True,
    "belongs_to_collection": False,
    "cast_size": 22,
    "crew_size": 35,
    "has_homepage": False,
    "has_tagline": True
}

feat_row_A = prepare_single_movie_features(movie_input_A, partition_stats)
pred_rev_A = predict_movie_revenue(champion_res["pipeline"], feat_row_A)
multiple_A = pred_rev_A / movie_input_A["budget"] if movie_input_A["budget"] > 0 else 0

print("=== Prediction Scenario A: Mid-Budget Action Film ===")
print(f"Budget:                ${movie_input_A['budget']:,}")
print(f"Predicted Revenue:     ${pred_rev_A:,.0f}")
print(f"Gross Revenue Multiple: {multiple_A:.2f}x budget")

# Test Scenario B: Zero-Budget Independent Movie
movie_input_B = {
    "budget": 0,
    "runtime": 95,
    "release_date": "2026-10-31",
    "genres": ["Horror", "Mystery"],
    "original_language": "en",
    "production_country": "US",
    "company_count": 1,
    "is_major_studio": False,
    "belongs_to_collection": False,
    "cast_size": 8,
    "crew_size": 12,
    "has_homepage": False,
    "has_tagline": False
}

feat_row_B = prepare_single_movie_features(movie_input_B, partition_stats)
pred_rev_B = predict_movie_revenue(champion_res["pipeline"], feat_row_B)

print("\n=== Prediction Scenario B: Zero-Budget Indie Movie ===")
print("Budget:                $0 (Unrecorded / Missing)")
print(f"Imputed Training Median: ${partition_stats['median_budget']:,}")
print(f"Predicted Revenue:     ${pred_rev_B:,.0f}")
print(f"Zero-budget indicator: {feat_row_B['budget_is_zero'].iloc[0]} (Properly flagged in feature vector)")

=== Prediction Scenario A: Mid-Budget Action Film ===
Budget:                $45,000,000
Predicted Revenue:     $48,919,613
Gross Revenue Multiple: 1.09x budget

=== Prediction Scenario B: Zero-Budget Indie Movie ===
Budget:                $0 (Unrecorded / Missing)
Imputed Training Median: $15,000,000.0
Predicted Revenue:     $265,255
Zero-budget indicator: 1 (Properly flagged in feature vector)


---
## Section 15: Final Model Training on 100% of Labeled Data

Following rigorous validation and architecture selection, we retrain our champion model across **100% of available labeled training movies** to maximize sample efficiency before generating test predictions.

In [19]:
# 1. Derive final production statistics across 100% of labeled training data
final_production_stats = compute_training_stats(clean_train_raw)
print("Final Production Training Statistics:")
for k, v in final_production_stats.items():
    print(f"  {k}: {v}")

# 2. Extract full feature matrix and target
X_full, y_full, _ = engineer_movie_features(clean_train_raw, is_train=True, stats=final_production_stats)
assert_zero_target_leakage(X_full)

# 3. Construct and fit full production pipeline
final_preprocessor, _, _ = build_preprocessor(list(X_full.columns))

# Extract champion hyperparameters
champ_hyperparams = {}
for m_name, hparams in models_to_train:
    if m_name == champion_key:
        champ_hyperparams = hparams
        break

final_estimator = create_model(champion_key, champ_hyperparams, random_state=RANDOM_STATE)
final_pipeline = Pipeline([
    ("preprocessor", final_preprocessor),
    ("estimator", final_estimator)
])

y_full_log = np.log1p(np.clip(y_full.values, 0, None))

t0 = time.time()
final_pipeline.fit(X_full, y_full_log)
fit_time_full = round(time.time() - t0, 3)

print(f"\nFinal Production Model ({champion_key}) fitted on {len(X_full):,} movies in {fit_time_full}s.")

Final Production Training Statistics:
  median_budget: 16000000.0
  median_runtime: 104.0
  budget_p99: 200000000.0
  runtime_min: 11.0
  runtime_max: 338.0
  median_release_year: 2004
  median_release_month: 7
  median_release_dayofweek: 4


Target Leakage Audit PASSED: 0 prohibited columns found among 37 engineered features.



Final Production Model (extra_trees) fitted on 3,000 movies in 0.538s.


---
## Section 16: Kaggle Test Predictions and Submission

We implement `generate_submission()` to process the unlabelled test set (`test.csv`, 4,398 movies), predict worldwide box office revenues, and export a verified `submission.csv`.

In [20]:
def generate_submission(
    pipeline: Pipeline,
    test_raw: pd.DataFrame,
    stats: Dict[str, Any],
    output_path: str = "submission.csv"
) -> pd.DataFrame:
    """
    Generates test predictions and saves a verified submission file.
    """
    # 1. Feature Engineering on test data reusing final training statistics
    X_test_matrix, _, _ = engineer_movie_features(test_raw, is_train=False, stats=stats)
    
    # 2. Predict in log-space and convert back to USD
    test_log_preds = pipeline.predict(X_test_matrix)
    test_preds = np.clip(np.expm1(test_log_preds), 0, None)
    
    # 3. Format submission DataFrame preserving test IDs
    sub_df = pd.DataFrame({
        "id": test_raw["id"].values,
        "revenue": np.round(test_preds, 2)
    })
    
    # 4. Strict assertions
    assert len(sub_df) == len(test_raw), f"Row count mismatch! {len(sub_df)} vs {len(test_raw)}"
    assert list(sub_df.columns) == ["id", "revenue"], f"Columns mismatch: {sub_df.columns}"
    assert not sub_df.isna().any().any(), "Submission contains NaN values!"
    assert (sub_df["revenue"] >= 0).all(), "Submission contains negative values!"
    
    # 5. Export to disk
    sub_df.to_csv(output_path, index=False)
    return sub_df

submission_df = generate_submission(final_pipeline, test_df, final_production_stats, output_path="submission.csv")

print(f"Kaggle Submission successfully exported to 'submission.csv'")
print(f"Total Rows:    {len(submission_df):,} (Exactly matches test.csv row count: {len(test_df):,})")
print(f"Columns:       {list(submission_df.columns)}")
print(f"Revenue Range: ${submission_df['revenue'].min():,.2f} to ${submission_df['revenue'].max():,.2f}")

print("\nFirst 10 Test Predictions:")
display(submission_df.head(10))

Kaggle Submission successfully exported to 'submission.csv'
Total Rows:    4,398 (Exactly matches test.csv row count: 4,398)
Columns:       ['id', 'revenue']
Revenue Range: $10.32 to $1,182,592,901.08

First 10 Test Predictions:


,id,revenue
0,3001,6456025.34
1,3002,2830766.01
2,3003,9442642.79
3,3004,1835704.76
4,3005,811607.99
5,3006,1635374.31
6,3007,2113229.13
7,3008,16669553.54
8,3009,18566638.28
9,3010,245553758.88


---
## Section 17: Save the Final Model

We bundle the fitted Scikit-Learn pipeline, training imputation statistics, and feature schemas into a single portable binary using `joblib.dump()`.

In [21]:
# Prepare model artifact bundle
model_bundle = {
    "pipeline": final_pipeline,
    "training_stats": final_production_stats,
    "feature_columns": ALL_FEATURE_COLUMNS,
    "model_name": champion_key,
    "hyperparameters": champ_hyperparams,
    "saved_timestamp": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}

model_save_path = "best_movie_revenue_model.joblib"
joblib.dump(model_bundle, model_save_path)
print(f"Model bundle successfully saved to '{model_save_path}' ({os.path.getsize(model_save_path) / 1024:.1f} KB).")

# Verification: Load the model back and run inference on test row
loaded_bundle = joblib.load(model_save_path)
loaded_pipeline = loaded_bundle["pipeline"]
loaded_stats = loaded_bundle["training_stats"]

sample_verify = predict_movie_revenue(loaded_pipeline, feat_row_A)
print(f"Loaded model verification test: Predicted ${sample_verify:,.0f} (Matches original: ${pred_rev_A:,.0f})")
print("Model serialization and deserialization integrity confirmed.")

Model bundle successfully saved to 'best_movie_revenue_model.joblib' (7570.1 KB).


Loaded model verification test: Predicted $56,213,153 (Matches original: $48,919,613)
Model serialization and deserialization integrity confirmed.


---
## Section 18: Final Findings, Limitations & Industry Reality

### 18.1 Empirical Findings Summary
* **Log-Budget Dominance:** Production budget is overwhelmingly the most predictive single pre-release feature. Because budget represents executive confidence and resource allocation, it strongly sets the scale of theatrical distribution.
* **Missing Budget Nuance:** 27% of movies lacked reported budgets. Explicitly indicating `budget_is_zero` alongside median imputation allowed tree models to distinguish unrecorded low-budget titles without artificially biasing predictions.
* **Franchise & Studio Leverage:** Membership in a franchise collection (`belongs_to_collection`) and backing by a major Hollywood studio (`is_major_studio`) significantly raise median box-office floors.

### 18.2 Inherent Limitations of Pre-Release Box Office Prediction
1. **Theatrical Rental Split:** Gross box office is not studio profit. Theatrical exhibitors typically retain 40%–55% of domestic ticket sales and up to 60%–75% of international ticket sales.
2. **Missing P&A (Print & Advertising) Data:** Marketing budgets often equal or exceed physical production budgets for major tentpoles (\$100M–\$150M in P&A), but are entirely absent from the TMDB dataset.
3. **Black Swan Dynamics:** Word-of-mouth momentum, critical reception, social media viral trends, and counter-programming dynamics occur post-release and cannot be fully anticipated solely from pre-release metadata.